# Week 2: Bonus Exercise - Threshold Tuning Deep Dive

**Level:** Advanced  
**Scaffolding:** 30% provided (you'll write most of the code!)  
**Time Estimate:** 45-60 minutes  
**Prerequisites:** Complete `week2_postclass_exercise.ipynb` first

---

## Learning Objectives

By completing this bonus exercise, you will:
1. Understand how changing the probability threshold affects performance
2. Create precision-recall curves
3. Find optimal thresholds for specific business objectives
4. Implement custom threshold logic in production-ready code
5. Analyze tradeoffs quantitatively

---

## The Challenge

You have a heart disease classifier with default threshold=0.5. Your stakeholders have different priorities:

**Scenario A - Primary Care Screening:**
- Goal: Don't miss ANY heart disease cases (maximize recall)
- Acceptable: More false alarms (follow-up tests are cheap)

**Scenario B - Insurance Risk Assessment:**
- Goal: Only flag high-confidence cases (maximize precision)
- Acceptable: Miss some borderline cases

**Scenario C - Balanced Approach:**
- Goal: Best balance of precision and recall (maximize F1)

**Your task:** Find the optimal threshold for each scenario.

---

## Setup

Run this cell to load the model and data from your post-class exercise.

In [ ]:
# Complete setup (copy from your post-class exercise)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, confusion_matrix, precision_score,
    recall_score, f1_score, precision_recall_curve,
    roc_curve, auc
)

%matplotlib inline

# Load and prepare data
column_names = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 
                'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target']

df = pd.read_csv(
    'https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data',
    names=column_names,
    na_values='?'
)

df['target'] = (df['target'] > 0).astype(int)
df = df.dropna()

X = df.drop('target', axis=1).values
y = df['target'].values

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Train model
model = LogisticRegression(max_iter=10000, random_state=42)
model.fit(X_train_scaled, y_train)

# Get probabilities
y_pred_proba = model.predict_proba(X_test_scaled)

print("✅ Setup complete!")
print(f"Test set size: {len(y_test)}")
print(f"Positive class (disease): {(y_test==1).sum()}")
print(f"Negative class (no disease): {(y_test==0).sum()}")

---

## Part 1: Exploring Different Thresholds

### Task 1.1: Create a Function to Evaluate Any Threshold

**TODO:** Write a function that takes a threshold and returns precision, recall, F1, and accuracy.

In [ ]:
def evaluate_threshold(y_true, y_pred_proba, threshold):
    """
    Evaluate model performance at a given threshold.
    
    Parameters:
    -----------
    y_true : array
        True labels
    y_pred_proba : array
        Predicted probabilities for positive class
    threshold : float
        Decision threshold (0 to 1)
    
    Returns:
    --------
    dict : Dictionary with precision, recall, f1, accuracy, and confusion matrix
    """
    # TODO: Convert probabilities to predictions using the threshold
    # Hint: y_pred = (y_pred_proba >= threshold).astype(int)
    
    y_pred = ____
    
    # TODO: Calculate all metrics
    # Hint: Use sklearn metric functions
    
    precision = ____
    recall = ____
    f1 = ____
    accuracy = ____
    cm = ____
    
    return {
        'threshold': threshold,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'accuracy': accuracy,
        'confusion_matrix': cm
    }

# Test your function
result = evaluate_threshold(y_test, y_pred_proba[:, 1], 0.5)
print("Results for threshold=0.5:")
print(f"  Precision: {result['precision']:.3f}")
print(f"  Recall:    {result['recall']:.3f}")
print(f"  F1:        {result['f1']:.3f}")
print(f"  Accuracy:  {result['accuracy']:.3f}")

### Task 1.2: Test Multiple Thresholds

**TODO:** Evaluate the model at thresholds from 0.1 to 0.9 and store results.

In [ ]:
# TODO: Test thresholds from 0.1 to 0.9 in steps of 0.05
# Hint: Use np.arange(0.1, 0.95, 0.05)
# Expected: List of dictionaries with results for each threshold

thresholds = ____
results = []

for threshold in thresholds:
    result = ____  # Evaluate this threshold
    results.append(result)

# Convert to DataFrame for easy analysis
results_df = pd.DataFrame(results)

print("✅ Evaluated all thresholds!")
print(f"\nFirst few results:")
print(results_df.head(10))

### Task 1.3: Visualize the Tradeoff

**TODO:** Plot how precision, recall, and F1 change with threshold.

In [ ]:
# TODO: Create a line plot showing precision, recall, and F1 vs threshold
# Hint: Use plt.plot() three times with different colors
# Expected: Three curves showing the tradeoff

plt.figure(figsize=(10, 6))

# Plot precision
plt.____(results_df['threshold'], results_df['____'], 
         label='____', marker='o', color='____')

# Plot recall  
plt.____(results_df['threshold'], results_df['____'],
         label='____', marker='s', color='____')

# Plot F1
plt.____(results_df['threshold'], results_df['____'],
         label='____', marker='^', color='____', linewidth=2)

# Add vertical line at default threshold
plt.axvline(x=0.5, color='black', linestyle='--', alpha=0.5, label='Default (0.5)')

plt.xlabel('Decision Threshold')
plt.ylabel('Score')
plt.title('Precision-Recall-F1 Tradeoff vs Threshold')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Question 1.1:** Looking at the plot:

a) What happens to precision as threshold increases?

*Your answer:*

___________________________________________________________________

b) What happens to recall as threshold increases?

*Your answer:*

___________________________________________________________________

c) Where does F1 reach its maximum?

*Your answer:*

___________________________________________________________________

---

## Part 2: Finding Optimal Thresholds for Each Scenario

### Task 2.1: Scenario A - Maximize Recall (Primary Care Screening)

**Goal:** Find threshold that gives recall ≥ 0.95 with best possible precision.

**TODO:** Find the optimal threshold for Scenario A.

In [ ]:
# TODO: Filter results to only thresholds where recall >= 0.95
# Then find the one with highest precision
# Hint: Use results_df[results_df['recall'] >= 0.95]

high_recall_results = results_df[results_df['____'] >= ____]

if len(high_recall_results) > 0:
    # Find threshold with highest precision among high-recall options
    best_idx = high_recall_results['____'].idxmax()
    optimal_a = results_df.loc[best_idx]
    
    print("=" * 60)
    print("SCENARIO A: Primary Care Screening (Maximize Recall)")
    print("=" * 60)
    print(f"\nOptimal Threshold: {optimal_a['threshold']:.2f}")
    print(f"\nPerformance:")
    print(f"  Recall:    {optimal_a['recall']:.3f} ({optimal_a['recall']*100:.1f}%)")
    print(f"  Precision: {optimal_a['precision']:.3f} ({optimal_a['precision']*100:.1f}%)")
    print(f"  F1-Score:  {optimal_a['f1']:.3f}")
    print(f"  Accuracy:  {optimal_a['accuracy']:.3f}")
    print(f"\nInterpretation: Catch {optimal_a['recall']*100:.0f}% of disease cases")
else:
    print("⚠️  No threshold achieves recall >= 0.95. Try lowering the requirement.")

### Task 2.2: Scenario B - Maximize Precision (Insurance Risk)

**Goal:** Find threshold that gives precision ≥ 0.90 with best possible recall.

**TODO:** Find the optimal threshold for Scenario B.

In [ ]:
# TODO: Filter results to only thresholds where precision >= 0.90
# Then find the one with highest recall

high_precision_results = results_df[results_df['____'] >= ____]

if len(high_precision_results) > 0:
    best_idx = high_precision_results['____'].idxmax()
    optimal_b = results_df.loc[best_idx]
    
    print("=" * 60)
    print("SCENARIO B: Insurance Risk Assessment (Maximize Precision)")
    print("=" * 60)
    print(f"\nOptimal Threshold: {optimal_b['threshold']:.2f}")
    print(f"\nPerformance:")
    print(f"  Precision: {optimal_b['precision']:.3f} ({optimal_b['precision']*100:.1f}%)")
    print(f"  Recall:    {optimal_b['recall']:.3f} ({optimal_b['recall']*100:.1f}%)")
    print(f"  F1-Score:  {optimal_b['f1']:.3f}")
    print(f"  Accuracy:  {optimal_b['accuracy']:.3f}")
    print(f"\nInterpretation: {optimal_b['precision']*100:.0f}% of disease predictions are correct")
else:
    print("⚠️  No threshold achieves precision >= 0.90. Try lowering the requirement.")

### Task 2.3: Scenario C - Best F1 Score (Balanced)

**Goal:** Find threshold that maximizes F1 score.

**TODO:** Find the optimal threshold for Scenario C.

In [ ]:
# TODO: Find threshold with maximum F1 score
# Hint: Use results_df['f1'].idxmax()

best_f1_idx = results_df['____'].____
optimal_c = results_df.loc[best_f1_idx]

print("=" * 60)
print("SCENARIO C: Balanced Approach (Maximize F1)")
print("=" * 60)
print(f"\nOptimal Threshold: {optimal_c['threshold']:.2f}")
print(f"\nPerformance:")
print(f"  F1-Score:  {optimal_c['f1']:.3f}")
print(f"  Precision: {optimal_c['precision']:.3f} ({optimal_c['precision']*100:.1f}%)")
print(f"  Recall:    {optimal_c['recall']:.3f} ({optimal_c['recall']*100:.1f}%)")
print(f"  Accuracy:  {optimal_c['accuracy']:.3f}")
print(f"\nInterpretation: Best balance between precision and recall")

### Task 2.4: Compare All Three Scenarios

**TODO:** Create a comparison table.

In [ ]:
# TODO: Create comparison DataFrame
# Hint: Use pd.DataFrame with a list of dictionaries

comparison = pd.DataFrame([
    {
        'Scenario': 'A: Max Recall (Primary Care)',
        'Threshold': ____,
        'Precision': ____,
        'Recall': ____,
        'F1': ____
    },
    {
        'Scenario': 'B: Max Precision (Insurance)',
        'Threshold': ____,
        'Precision': ____,
        'Recall': ____,
        'F1': ____
    },
    {
        'Scenario': 'C: Max F1 (Balanced)',
        'Threshold': ____,
        'Precision': ____,
        'Recall': ____,
        'F1': ____
    },
    {
        'Scenario': 'Default (0.5)',
        'Threshold': 0.5,
        'Precision': results_df[results_df['threshold']==0.5]['precision'].values[0],
        'Recall': results_df[results_df['threshold']==0.5]['recall'].values[0],
        'F1': results_df[results_df['threshold']==0.5]['f1'].values[0]
    }
])

print("\n" + "=" * 80)
print("COMPARISON OF OPTIMAL THRESHOLDS")
print("=" * 80)
print(comparison.to_string(index=False))
print("=" * 80)

**Question 2.1:** 

a) Which scenario has the LOWEST threshold? Why does that make sense?

*Your answer:*

___________________________________________________________________

___________________________________________________________________

b) Which scenario has the HIGHEST threshold? Why does that make sense?

*Your answer:*

___________________________________________________________________

___________________________________________________________________

c) How much did we improve recall by tuning the threshold (Scenario A vs Default)?

*Your answer:*

___________________________________________________________________

---

## Part 3: Precision-Recall Curve

### Task 3.1: Create Precision-Recall Curve

**TODO:** Generate and plot the precision-recall curve.

In [ ]:
# TODO: Calculate precision-recall curve
# Hint: Use precision_recall_curve from sklearn

precision_curve, recall_curve, thresholds_curve = ____(
    ____,  # y_test
    ____   # y_pred_proba[:, 1]
)

# Plot precision-recall curve
plt.figure(figsize=(10, 7))
plt.plot(recall_curve, precision_curve, color='blue', lw=2, label='PR Curve')

# Mark the three optimal points
plt.scatter([optimal_a['recall']], [optimal_a['precision']], 
            color='red', s=100, marker='o', label='Scenario A (Max Recall)', zorder=5)
plt.scatter([optimal_b['recall']], [optimal_b['precision']], 
            color='green', s=100, marker='s', label='Scenario B (Max Precision)', zorder=5)
plt.scatter([optimal_c['recall']], [optimal_c['precision']], 
            color='purple', s=100, marker='^', label='Scenario C (Max F1)', zorder=5)

plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve with Optimal Points')
plt.legend()
plt.grid(alpha=0.3)
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.tight_layout()
plt.show()

# Calculate area under PR curve
pr_auc = auc(recall_curve, precision_curve)
print(f"\nArea Under PR Curve: {pr_auc:.3f}")

---

## Part 4: Production Implementation

### Task 4.1: Create a Custom Classifier with Tuned Threshold

**TODO:** Wrap the model in a class that uses your optimal threshold.

In [ ]:
class ThresholdTunedClassifier:
    """
    Wrapper for sklearn classifier with custom decision threshold.
    """
    def __init__(self, base_classifier, threshold=0.5):
        """
        Parameters:
        -----------
        base_classifier : sklearn classifier
            Must have predict_proba method
        threshold : float
            Decision threshold (0 to 1)
        """
        self.base_classifier = base_classifier
        self.threshold = threshold
    
    def fit(self, X, y):
        """Train the base classifier."""
        # TODO: Fit the base classifier
        self.base_classifier.____(____, ____)
        return self
    
    def predict_proba(self, X):
        """Get probability predictions."""
        # TODO: Return probabilities from base classifier
        return self.base_classifier.____(____ )
    
    def predict(self, X):
        """Make predictions using custom threshold."""
        # TODO: Get probabilities and apply custom threshold
        # Hint: proba >= threshold
        proba = self.____(____ )
        return (proba[:, 1] >= self.____).astype(int)

# Test it!
print("Testing ThresholdTunedClassifier...\n")

# Scenario A: Low threshold for high recall
classifier_a = ThresholdTunedClassifier(model, threshold=optimal_a['threshold'])
y_pred_a = classifier_a.predict(X_test_scaled)
print(f"Scenario A (threshold={optimal_a['threshold']:.2f}):")
print(f"  Recall: {recall_score(y_test, y_pred_a):.3f}")

# Scenario B: High threshold for high precision
classifier_b = ThresholdTunedClassifier(model, threshold=optimal_b['threshold'])
y_pred_b = classifier_b.predict(X_test_scaled)
print(f"\nScenario B (threshold={optimal_b['threshold']:.2f}):")
print(f"  Precision: {precision_score(y_test, y_pred_b):.3f}")

print("\n✅ Custom classifier working!")

---

## Part 5: Final Analysis

### Task 5.1: Quantify the Cost-Benefit Tradeoff

**Scenario:** You're presenting to hospital administrators.

**Costs:**
- Missed heart disease case (FN): $500,000 (lifetime treatment cost if caught late)
- False alarm (FP): $2,000 (cost of additional diagnostic tests)

**TODO:** Calculate total cost for each scenario.

In [ ]:
def calculate_cost(confusion_matrix, cost_fn=500000, cost_fp=2000):
    """
    Calculate total cost based on confusion matrix.
    
    Parameters:
    -----------
    confusion_matrix : array
        2x2 confusion matrix
    cost_fn : float
        Cost of false negative (missed disease)
    cost_fp : float
        Cost of false positive (false alarm)
    
    Returns:
    --------
    float : Total cost
    """
    # TODO: Extract FP and FN from confusion matrix
    # Hint: cm.ravel() gives tn, fp, fn, tp
    
    tn, fp, fn, tp = confusion_matrix.ravel()
    
    # TODO: Calculate total cost
    total_cost = (fn * ____) + (fp * ____)
    
    return total_cost, fp, fn

# Calculate costs for each scenario
print("\n" + "=" * 70)
print("COST-BENEFIT ANALYSIS")
print("=" * 70)
print(f"\nCost assumptions:")
print(f"  False Negative (missed disease): $500,000")
print(f"  False Positive (false alarm): $2,000")
print(f"\nResults for {len(y_test)} patients:")
print("-" * 70)

for name, result in [('Scenario A (Max Recall)', optimal_a),
                      ('Scenario B (Max Precision)', optimal_b),
                      ('Scenario C (Max F1)', optimal_c),
                      ('Default (0.5)', results_df[results_df['threshold']==0.5].iloc[0])]:
    cost, fp, fn = calculate_cost(result['confusion_matrix'])
    print(f"\n{name}:")
    print(f"  False Negatives: {fn} × $500,000 = ${fn * 500000:,}")
    print(f"  False Positives: {fp} × $2,000 = ${fp * 2000:,}")
    print(f"  TOTAL COST: ${cost:,}")

print("\n" + "=" * 70)

**Question 5.1:**

a) Which scenario has the LOWEST total cost?

*Your answer:*

___________________________________________________________________

b) How much money did we save by tuning the threshold (best scenario vs default)?

*Your answer:*

___________________________________________________________________

c) In your own words, explain why threshold tuning matters for business outcomes.

*Your answer:*

___________________________________________________________________

___________________________________________________________________

___________________________________________________________________

---

## 🎉 Congratulations!

You've completed the advanced threshold tuning exercise!

**What you mastered:**
1. ✅ Systematic threshold exploration
2. ✅ Precision-recall tradeoff analysis
3. ✅ Finding optimal thresholds for business objectives
4. ✅ Building custom threshold-tuned classifiers
5. ✅ Quantifying cost-benefit tradeoffs

**Key insights:**
- Default threshold (0.5) is rarely optimal for real-world problems
- Different business contexts require different threshold strategies
- Threshold tuning can save significant costs without retraining
- F1-maximizing threshold balances precision and recall
- Precision-recall curves visualize all possible tradeoffs

**Production tips:**
- Always test multiple thresholds in development
- Document your threshold choice and rationale
- Monitor performance if costs/priorities change
- Consider making threshold configurable
- Communicate tradeoffs clearly to stakeholders

---

## Extension Challenges

**If you want to go deeper:**

1. **Different cost ratios:** What if FN cost was $1M instead of $500K? How does the optimal threshold change?

2. **Constraint optimization:** Find threshold that minimizes FN while keeping FP below a fixed number (e.g., max 5 false positives).

3. **Multi-objective:** Use Pareto optimization to find thresholds that are "undominated" (can't improve one metric without hurting another).

4. **Threshold by subgroup:** Do different patient demographics need different thresholds?

5. **Dynamic thresholds:** Could the threshold adapt based on patient risk factors?

---

**Excellent work on this advanced exercise!** 🎓

---

*Week 2 Bonus Exercise v1.0 | December 2025*